# Utilities for OCR

In [12]:
def run_ocr(source_dir = 'cropped_imgs', result_dir = 'tesseract_result', model=None):

    if not model:
        raise ValueError("Model must be provided for OCR processing.")
    
    # Placeholder for OCR processing
    print("Running OCR on training and testing data...")
    import os

    # Iterate through all folders in the source directory
    for folder in os.listdir(source_dir):
        folder_path = os.path.join(source_dir, folder)
        if os.path.isdir(folder_path):
            # Create a folder with the same name in the result directory
            result_folder = os.path.join(result_dir, folder)
            if not os.path.exists(result_folder):
                os.makedirs(result_folder)
            # Iterate through all images in the folder
            for page in os.listdir(folder_path):
                if page != 'page_3':
                    continue
                page_path = os.path.join(folder_path, page)
                page_output = ""
                for image_file in os.listdir(page_path):
                    if image_file.endswith('.png'):
                        image_path = os.path.join(page_path, image_file)
                        
                        # Interact with the Gemini LLM for the image
                        response = model(image_path=image_path)
                        response = response.strip()
                        if response and '\n' != response[-1]:
                            response += '\n'
                        page_output += response
                        
                # Write the response to a separate text file for each page
                output_file = os.path.join(result_dir, folder,  f'{page.replace(".png","")}.txt')
                with open(output_file, 'w') as f:
                    f.write(page_output)
                    print('Saved:', output_file)

# TrOCR Part

## Load TrOCR Models

In [13]:
import os
import torch
from functools import partial
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from PIL import Image

In [14]:
def run_tr_ocr(image_path, processor, model):
    # Load image
    image = Image.open(image_path).convert("RGB")

    # Preprocess image
    pixel_values = processor(image, return_tensors="pt").pixel_values

    # Generate text
    generated_ids = model.generate(pixel_values)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

    return generated_text[0]

## Extract Text Using pretrained TrOCR

In [15]:
# Load processor and model.
# Choose either "microsoft/trocr-base-handwritten" or "microsoft/trocr-base-printed" as needed.
processor = TrOCRProcessor.from_pretrained('qantev/trocr-base-spanish')
model = VisionEncoderDecoderModel.from_pretrained('qantev/trocr-base-spanish')

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.50.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_mod

In [16]:
run_ocr(source_dir = 'cropped_imgs', 
        result_dir = 'trocr_result', 
        model=partial(run_tr_ocr, processor=processor, model=model))

Running OCR on training and testing data...


/home/chanelsi/.conda/envs/renaissance/lib/python3.11/site-packages/transformers/generation/utils.py:1581: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Saved: trocr_result/Buendia - Instruccion/page_3.txt
Saved: trocr_result/Constituciones sinodales Calahorra 1602/page_3.txt
Saved: trocr_result/Ezcaray - Vozes/page_3.txt
Saved: trocr_result/Mendo - Principe perfecto/page_3.txt
Saved: trocr_result/PORCONES.228.35 – 1636/page_3.txt


## Fine tune TrOCR

In [17]:
# Load processor and model.
processor = TrOCRProcessor.from_pretrained('qantev/trocr-base-spanish')
model = VisionEncoderDecoderModel.from_pretrained('qantev/trocr-base-spanish')

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.50.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_mod

In [21]:
# Update model configuration with special token IDs.
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size  # Set correct vocab size

In [22]:
# Define a custom dataset that pairs images and labels based on matching filenames.
class MyOCRDataset(Dataset):
    def __init__(self, images_dir, labels_dir, processor, max_length=128):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.processor = processor
        self.max_length = max_length
        self.image_files = sorted(os.listdir(images_dir))
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Get image file and corresponding label file (assume same basename, .txt extension)
        image_filename = self.image_files[idx]
        image_path = os.path.join(self.images_dir, image_filename)
        label_filename = os.path.splitext(image_filename)[0] + ".txt"
        label_path = os.path.join(self.labels_dir, label_filename)
        
        # Read the label text
        with open(label_path, "r", encoding="utf-8") as f:
            text = f.read().strip()
        
        # Load image and convert to RGB
        image = Image.open(image_path).convert("RGB")
        
        # Use the processor to get inputs for the model.
        # Here we assume that the processor can handle both image and text in one call.
        encoding = self.processor(image, text, 
                                  padding="max_length", 
                                  truncation=True, 
                                  max_length=self.max_length, 
                                  return_tensors="pt")
        # Remove batch dimension
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        return encoding

In [23]:
# Prepare your dataset (adjust paths as necessary)
dataset = MyOCRDataset(
    images_dir="train_data/images",
    labels_dir="train_data/labels",
    processor=processor
)

In [24]:
# Set up training arguments.
training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-finetuned",
    per_device_train_batch_size=8,
    num_train_epochs=15,
    predict_with_generate=True,
    logging_steps=10,
    save_steps=100,
)


In [25]:
# Initialize Trainer. Note that for image input you pass the processor's feature extractor.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=processor.feature_extractor,
)

/home/chanelsi/.conda/envs/renaissance/lib/python3.11/site-packages/transformers/models/trocr/processing_trocr.py:152: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(
/tmp/ipykernel_1757317/1141205744.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
# Start training.
trainer.train()

In [ ]:
# 1. Save the model to a local directory
output_dir = "trocr-finetuned-model"
trainer.save_model(output_dir)
print(f"Model saved to {output_dir}")

# 2. If you have a tokenizer, save it as well
# if 'tokenizer' in globals():
#     tokenizer.save_pretrained(output_dir)
#     print(f"Tokenizer saved to {output_dir}")

## Extract Text Using fine-tuned TrOCR

In [33]:
import os
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq

# Load from local directory
# processor = TrOCRProcessor.from_pretrained("./trocr-finetuned-model")
processor = TrOCRProcessor.from_pretrained('qantev/trocr-base-spanish')
model = VisionEncoderDecoderModel.from_pretrained("./trocr-finetuned-model-405")

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.50.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_mod

In [34]:
run_ocr(source_dir = 'cropped_imgs', 
        result_dir = 'trocr_ft_result-405', 
        model=partial(run_tr_ocr, processor=processor, model=model))

Running OCR on training and testing data...
Saved: trocr_ft_result-405/Buendia - Instruccion/page_3.txt
Saved: trocr_ft_result-405/Constituciones sinodales Calahorra 1602/page_3.txt
Saved: trocr_ft_result-405/Ezcaray - Vozes/page_3.txt
Saved: trocr_ft_result-405/Mendo - Principe perfecto/page_3.txt
Saved: trocr_ft_result-405/PORCONES.228.35 – 1636/page_3.txt


# Extract Text using DONUT

In [50]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
import re

In [51]:
# 1. Load model and processor
model_name = "pandafm/donut-es"
model = VisionEncoderDecoderModel.from_pretrained(model_name)
processor = DonutProcessor.from_pretrained(model_name)


Config of the encoder: <class 'transformers.models.donut.modeling_donut_swin.DonutSwinModel'> is overwritten by shared encoder config: DonutSwinConfig {
  "attention_probs_dropout_prob": 0.0,
  "depths": [
    2,
    2,
    14,
    2
  ],
  "drop_path_rate": 0.1,
  "embed_dim": 128,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 1024,
  "image_size": [
    1410,
    878
  ],
  "initializer_range": 0.02,
  "layer_norm_eps": 1e-05,
  "mlp_ratio": 4.0,
  "model_type": "donut-swin",
  "num_channels": 3,
  "num_heads": [
    4,
    8,
    16,
    32
  ],
  "num_layers": 4,
  "patch_size": 4,
  "path_norm": true,
  "qkv_bias": true,
  "torch_dtype": "float32",
  "transformers_version": "4.50.3",
  "use_absolute_embeddings": false,
  "window_size": 10
}

Config of the decoder: <class 'transformers.models.mbart.modeling_mbart.MBartForCausalLM'> is overwritten by shared decoder config: MBartConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cro

In [60]:
def run_donut_ocr(image_path, processor, model):
    # Load image
    image = Image.open(image_path).convert("RGB")

    # Preprocess image
    pixel_values = processor(image, return_tensors="pt").pixel_values

    # Generate text
    task_prompt = "<s_cord-v2>"
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
    # decoder_input_ids = decoder_input_ids.to(device)

    # autoregressively generate sequence
    result = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=model.decoder.config.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )
    seq = processor.batch_decode(result.sequences)[0]
    seq = seq.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    seq = re.sub(r"<.*?>", "", seq, count=1).strip()  # remove first task start token
    seq = processor.token2json(seq)

    def extract_nm(data):
        results = []
        if isinstance(data, dict):
            for key, value in data.items():
                if key == "nm":
                    results.append(value)
                results.extend(extract_nm(value))
        elif isinstance(data, list):
            for item in data:
                results.extend(extract_nm(item))
        return results

    texts = extract_nm(seq)
    print(texts)
    texts = "\n".join([text for text in texts if type(text) == str and text.strip()])
    return texts

In [61]:
run_ocr(source_dir = 'uncropped_imgs', 
        result_dir = 'donut_result', 
        model=partial(run_donut_ocr, processor=processor, model=model))

Running OCR on training and testing data...
['D. Balthalar de Baftero y Lledo, Obifpoi', 'Huendia, Colegial que fue en el Imperial', 'Cordeillas, 8cc. Y brevemente digo', 'huenas coltumbres, fino que muy atento', 'Author con entrambas, deferibe,yen-', 'fena tan culta, y difereta la Virtud, coj', {'price': '2,50'}, 'muy atentos, mereceran, quando hombres,', 'aver nacido Senores. Porque no tolo fa-', 'bran fer Caballoros, fino tambien']
Saved: donut_result/Buendia - Instruccion/page_3.txt
['CedroMalopriagracidade.Dien', 'deCalachora, y LaCal cada, del Conlejo deCalachora y La Cal cada, del Conlejo deMache/Lad.&c. AlDean yCaul', 'defigurar', 'Capillos de las yeldas', 'Capillos de las yelchades y luradores de Cohrados, Hero', 'Cierigos,Mayordomos, Almitudoresde Cohradias, Her', 'mandates desonfades, y velinas, quanto a lo cipirroa, Y otras_quale/quinperaboalsa', 'otras_quale/quinperaboalsa', 'Las las ciudadanos, villas, y lugares detenetro otra-qual']
Saved: donut_result/Constituciones sin

# Evaluation

In [35]:
import os
import jiwer
import pandas as pd
import unicodedata

# Function to remove diacritical marks except for ñ (and Ñ)
def remove_accents_except_n(text):
    result = ""
    for char in text:
        if char in "ñÑ":
            result += char
        else:
            decomposed = unicodedata.normalize('NFD', char)
            filtered = ''.join(c for c in decomposed if unicodedata.category(c) != 'Mn')
            result += filtered
    return result

def remove_duplicate_whitespace(text):
    return ' '.join(text.split())

def convert_uv_to_u(text):
    text = text.replace('v', 'u')
    return text

def generate_result(output_dir, reference_dir = 'pdf_labels'):
    print(f'Evaluating {output_dir} against {reference_dir}')
    # Initialize lists to store WER and CER results
    results = []
    
    # Iterate through all folders in the reference directory
    for folder in os.listdir(reference_dir):
        folder_path = os.path.join(reference_dir, folder)
        if os.path.isdir(folder_path):
            # Iterate through all text files in the folder
            for ref_file in os.listdir(folder_path):
                if ref_file.endswith('.txt'):
                    ref_path = os.path.join(folder_path, ref_file)
                    out_path = os.path.join(output_dir, folder, ref_file)
    
                    # Read the reference text
                    with open(ref_path, 'r') as f:
                        ref_text = f.read()
    
                    # Read the output text
                    if os.path.exists(out_path):
                        with open(out_path, 'r') as f:
                            out_text = f.read()
    
                        # Apply the custom transformation
                        ref_text = remove_accents_except_n(ref_text)
                        out_text = remove_accents_except_n(out_text)
                        
                        # Remove duplicate whitespaces
                        ref_text = remove_duplicate_whitespace(ref_text)
                        out_text = remove_duplicate_whitespace(out_text)

                        # Convert 'v' to 'u'
                        ref_text = convert_uv_to_u(ref_text)
                        out_text = convert_uv_to_u(out_text)
    
                        # Calculate WER and CER using jiwer
                        wer = jiwer.wer(ref_text, out_text)
                        cer = jiwer.cer(ref_text, out_text)
    
                        # Print the results
                        print(f'{folder}/{ref_file} - WER: {wer:.4f}, CER: {cer:.4f}')
    
                        # Store the results in the list
                        results.append({'file': f'{folder}/{ref_file}', 'WER': wer, 'CER': cer})
    
    # Create a pandas DataFrame from the results
    df = pd.DataFrame(results)
    
    # Write the DataFrame to a CSV file
    df.to_csv(f'{output_dir}.csv', index=False)
    return df

In [36]:
df = generate_result('trocr_ft_result-81', reference_dir = 'pdf_labels')
print("Average WER:", df['WER'].mean())
print("Average CER:", df['CER'].mean())

Evaluating trocr_ft_result-81 against pdf_labels
Buendia - Instruccion/page_3.txt - WER: 0.9541, CER: 0.7253
Constituciones sinodales Calahorra 1602/page_3.txt - WER: 0.9366, CER: 0.6470
Ezcaray - Vozes/page_3.txt - WER: 0.9521, CER: 0.7014
Mendo - Principe perfecto/page_3.txt - WER: 0.9643, CER: 0.7115
PORCONES.228.35 – 1636/page_3.txt - WER: 0.9571, CER: 0.6986
Average WER: 0.9528507278267944
Average CER: 0.6967455131214175


In [37]:
df = generate_result('trocr_ft_result-405', reference_dir = 'pdf_labels')
print("Average WER:", df['WER'].mean())
print("Average CER:", df['CER'].mean())

Evaluating trocr_ft_result-405 against pdf_labels
Buendia - Instruccion/page_3.txt - WER: 0.9450, CER: 0.7270
Constituciones sinodales Calahorra 1602/page_3.txt - WER: 0.9284, CER: 0.6521
Ezcaray - Vozes/page_3.txt - WER: 0.9281, CER: 0.6957
Mendo - Principe perfecto/page_3.txt - WER: 0.9563, CER: 0.7069
PORCONES.228.35 – 1636/page_3.txt - WER: 0.9571, CER: 0.6910
Average WER: 0.9429852521305115
Average CER: 0.6945235173028733


In [38]:
df = generate_result('trocr_result', reference_dir = 'pdf_labels')
print("Average WER:", df['WER'].mean())
print("Average CER:", df['CER'].mean())

Evaluating trocr_result against pdf_labels
Buendia - Instruccion/page_3.txt - WER: 0.9174, CER: 0.7270
Constituciones sinodales Calahorra 1602/page_3.txt - WER: 0.7824, CER: 0.5586
Ezcaray - Vozes/page_3.txt - WER: 0.8802, CER: 0.6980
Mendo - Principe perfecto/page_3.txt - WER: 0.8810, CER: 0.6469
PORCONES.228.35 – 1636/page_3.txt - WER: 0.8767, CER: 0.6404
Average WER: 0.8675335687587367
Average CER: 0.6541654126730818


In [67]:
df = generate_result('donut_result', reference_dir = 'pdf_labels')
print("Average WER:", df['WER'].mean())
print("Average CER:", df['CER'].mean())

Evaluating donut_result against pdf_labels
Buendia - Instruccion/page_3.txt - WER: 0.9541, CER: 0.7270
Constituciones sinodales Calahorra 1602/page_3.txt - WER: 0.9559, CER: 0.8400
Ezcaray - Vozes/page_3.txt - WER: 0.9341, CER: 0.9219
Mendo - Principe perfecto/page_3.txt - WER: 0.9762, CER: 0.9574
PORCONES.228.35 – 1636/page_3.txt - WER: 0.9196, CER: 0.8406
Average WER: 0.9479889127349154
Average CER: 0.8573686274812726
